In [1]:
import importlib
import subprocess
import sys
from typing import Optional


def _ensure(import_name: str, pip_name: Optional[str] = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except Exception as e:
            raise ModuleNotFoundError(
                f"Missing dependency '{import_name}'. Tried to install '{pkg}' via pip but it failed. "
                f"Please install it in this environment (e.g. `pip install {pkg}`) and rerun. Original error: {e}"
            )


_ensure("datasets")
_ensure("pandas")
_ensure("matplotlib")
_ensure("seaborn")

from datasets import load_dataset

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display  # type: ignore
except Exception:  # pragma: no cover

    def display(x):
        print(x)

DATASET_NAME = "TIGER-Lab/WebInstruct-verified"
SAMPLE_N = 1000
SAMPLE_SEED = 42

# Load dataset (handles DatasetDict vs Dataset)
ds = load_dataset(DATASET_NAME)
if hasattr(ds, "keys"):
    splits = list(ds.keys())
    dsets = [ds[s] for s in splits]
    d_all = pd.concat([d.to_pandas() for d in dsets], ignore_index=True)
else:
    splits = ["(single)"]
    d_all = ds.to_pandas()

# Random subsample to SAMPLE_N rows for faster iteration
if len(d_all) > SAMPLE_N:
    d_all = d_all.sample(n=SAMPLE_N, random_state=SAMPLE_SEED).reset_index(drop=True)

print(f"Dataset: {DATASET_NAME}")
print(f"Splits loaded: {splits}")
print(f"Total rows (after subsample): {len(d_all)}  (seed={SAMPLE_SEED})")
print(f"Columns: {list(d_all.columns)}")

display(d_all.head(10))


/home/mjgwak/miniconda3/envs/self_critique/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset: TIGER-Lab/WebInstruct-verified
Splits loaded: ['train', 'test', 'train_legacy']
Total rows (after subsample): 1000  (seed=42)
Columns: ['id', 'question', 'answer', 'answer_type', 'category', 'difficulty']


,id,question,answer,answer_type,category,difficulty
0,1433007,"A charge +4q is placed at x = 0, and a charge ...",x = 2.15 units,Float,Physics,Senior High School
1,1921384,Aspirin is the common name for acetylsalicylic...,3.0,Float,Chemistry,Senior High School
2,565825,Who signed the 1969 petition to pay Eldridge C...,"The petition signers' names are not provided, ...",String,History,Senior High School
3,1638769,The differential equation $$(1-x^2)y''-xy'+p^2...,T_2(x) = 2x^2-1,Expression,Mathematics,University
4,4147,To heat {eq}\displaystyle\n\n20.3 g\n\n{/eq} o...,900.743 J,Float,Physics,Junior High School
5,2004325,Voltage (V)Time (s)\n1.27900.01624\n2.60000.03...,37.364 V/s,Float,Physics,Senior High School
6,731969,Calculate the minimum energy required to remov...,122.4 eV,Float,Physics,University
7,1876014,Find the first four non-zero terms of the Tayl...,1 + x/2 - (17/8)x^2 - (15/16)x^3,Expression,Mathematics,University
8,1326184,"Two cylindrical bars, each with a diameter of ...",4.15e-4 Ω,Float,Physics,Senior High School
9,1368554,One end of a 48 cm long copper rod with a diam...,145 W,Float,Physics,Senior High School


In [2]:
import os
from pathlib import Path

import pandas as pd
from datasets import load_dataset

os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

N_KEEP = 30
KEEP_SEED = 123

AIME_REPO = "talzoomanzoo/aime26"

USER_SUFFIX = "\n\nPresent the answer in LaTex format: \\boxed{Your answer}"

if "d_all" not in globals():
    raise RuntimeError("Run the cell above first to populate `d_all` (1k-subsampled member frame).")

cols = set(d_all.columns)
if {"dataset", "prompt", "ground_truth"}.issubset(cols):
    member_frame = d_all
elif {"question", "answer"}.issubset(cols):
    ds_label = str(globals().get("DATASET_NAME", "TIGER-Lab/WebInstruct-verified"))
    if "category" in d_all.columns:
        dataset_col = ds_label + "/" + d_all["category"].astype(str)
    else:
        dataset_col = ds_label
    member_frame = pd.DataFrame(
        {
            "dataset": dataset_col,
            "prompt": d_all["question"],
            "ground_truth": d_all["answer"],
        }
    )
else:
    raise KeyError(
        "Member frame needs Dolci columns {dataset, prompt, ground_truth} "
        "or WebInstruct columns {question, answer}. "
        f"Has: {list(d_all.columns)}"
    )

d30 = member_frame.sample(n=N_KEEP, random_state=KEEP_SEED).reset_index(drop=True)

aime_raw = load_dataset(AIME_REPO)
aime_split = aime_raw["train"] if hasattr(aime_raw, "keys") and "train" in aime_raw else (
    aime_raw[list(aime_raw.keys())[0]] if hasattr(aime_raw, "keys") else aime_raw
)
aime_df = aime_split.to_pandas()

need_cols = {"data_source", "prompt", "answer"}
missing_aime = need_cols - set(aime_df.columns)
if missing_aime:
    raise KeyError(
        f"{AIME_REPO} missing expected columns {sorted(missing_aime)}. Has: {list(aime_df.columns)}"
    )

def _extract_system_msg(prompt) -> str:
    """Extract the system-role content from a chat prompt (list/ndarray of dicts)."""
    if prompt is None:
        return ""
    try:
        items = list(prompt)
    except TypeError:
        return ""
    for item in items:
        if not isinstance(item, dict):
            continue
        if item.get("role") == "system" and item.get("content"):
            return str(item["content"])
    for item in items:
        if isinstance(item, dict) and item.get("content"):
            return str(item["content"])
    return ""


SYSTEM_MSG = _extract_system_msg(aime_df.iloc[0]["prompt"])
if not SYSTEM_MSG:
    raise RuntimeError(
        f"Could not extract a non-empty system message from {AIME_REPO}. "
        f"First prompt was: {aime_df.iloc[0]['prompt']!r}"
    )
print(f"Lifted system message from {AIME_REPO} ({len(SYSTEM_MSG)} chars)")


def _wrap_prompt(text) -> list[dict]:
    user_content = str(text).rstrip() + USER_SUFFIX
    return [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": user_content},
    ]


def _norm_answer(x) -> str | None:
    if x is None:
        return None
    s = str(x).strip()
    return s if s else None


member_rows = pd.DataFrame(
    {
        "data_source": d30["dataset"].astype(str),
        "prompt": [_wrap_prompt(p) for p in d30["prompt"].tolist()],
        "answer": [_norm_answer(a) for a in d30["ground_truth"].tolist()],
    }
)
member_rows["member"] = 1

aime_rows = aime_df[["data_source", "prompt", "answer"]].copy()
aime_rows["answer"] = [_norm_answer(a) for a in aime_rows["answer"].tolist()]
aime_rows["member"] = 0

combined = pd.concat([aime_rows, member_rows], ignore_index=True)

print(f"aime26 rows (member=0): {len(aime_rows)}")
print(f"member rows (member=1): {len(member_rows)}")
print(f"combined rows: {len(combined)}")
print(f"combined columns: {list(combined.columns)}")

BASE_DIR = Path(".") if Path("filter_benchmark_reasoner.ipynb").exists() else Path("benchmarks")
OUT_DIR = BASE_DIR / "filtered_samples"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_parquet = OUT_DIR / "talzoomanzoo__aime26_plus_WebInstruct-verified__with_member.parquet"
combined.to_parquet(out_parquet, index=False)
print(f"Wrote: {out_parquet}")

GENERALREASONER_DIR = BASE_DIR / "GENERALREASONER"
GENERALREASONER_DIR.mkdir(parents=True, exist_ok=True)
reasoner_member_parquet = GENERALREASONER_DIR / "reasoner_member.parquet"
combined.to_parquet(reasoner_member_parquet, index=False)
print(f"Wrote: {reasoner_member_parquet}")

display(combined.head(3))
display(combined.tail(3))


Lifted system message from talzoomanzoo/aime26 (385 chars)
aime26 rows (member=0): 60
member rows (member=1): 30
combined rows: 90
combined columns: ['data_source', 'prompt', 'answer', 'member']
Wrote: filtered_samples/talzoomanzoo__aime26_plus_WebInstruct-verified__with_member.parquet
Wrote: GENERALREASONER/reasoner_member.parquet


,data_source,prompt,answer,member
0,aime26,[{'content': ' When tackling complex reasoning...,277,0
1,aime26,[{'content': ' When tackling complex reasoning...,62,0
2,aime26,[{'content': ' When tackling complex reasoning...,79,0


,data_source,prompt,answer,member
87,TIGER-Lab/WebInstruct-verified/Business,"[{'role': 'system', 'content': ' When tackling...",b,1
88,TIGER-Lab/WebInstruct-verified/Mathematics,"[{'role': 'system', 'content': ' When tackling...",5488√3/9,1
89,TIGER-Lab/WebInstruct-verified/Mathematics,"[{'role': 'system', 'content': ' When tackling...",1 + x/2 - (17/8)x^2 - (15/16)x^3,1


In [3]:
import importlib
import os
import subprocess
import sys


def _ensure(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
from huggingface_hub import HfApi

try:
    from huggingface_hub import get_token as _hf_get_token  # huggingface_hub >= 0.19
except ImportError:
    from huggingface_hub import HfFolder  # legacy

    def _hf_get_token():
        return HfFolder.get_token()


HF_REPO_ID = "talzoomanzoo/reasoner_member"

if "out_parquet" not in globals():
    raise RuntimeError(
        "Run the merge cell above first; `out_parquet` is not defined."
    )

token = _hf_get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `huggingface-cli login` or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
url = api.upload_file(
    path_or_fileobj=str(out_parquet),
    path_in_repo=out_parquet.name,
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message=f"Add {out_parquet.name} (aime26 + TIGER-Lab/WebInstruct-verified, with member column)",
)
print(f"Uploaded to: {url}")


Processing Files (1 / 1): 100%|██████████| 18.5kB / 18.5kB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Uploaded to: https://huggingface.co/datasets/talzoomanzoo/reasoner_member/commit/2fd1ceea94796f9e69a1b57684826da3c1873c83


In [4]:
import os
from pathlib import Path

import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset

OLMOE_MEMBER_REPO = "talzoomanzoo/reasoner_member"
EURUS_RL_REPO = "talzoomanzoo/eurus_rl_train_test"
EXCLUDE_DATA_SOURCES = {"numina_amc_aime", "aime26"}

olmoe_raw = load_dataset(OLMOE_MEMBER_REPO)
olmoe_split_name = (
    "train"
    if isinstance(olmoe_raw, DatasetDict) and "train" in olmoe_raw
    else (list(olmoe_raw.keys())[0] if isinstance(olmoe_raw, DatasetDict) else None)
)
olmoe = olmoe_raw[olmoe_split_name] if olmoe_split_name else olmoe_raw

need_cols = {"data_source", "prompt", "answer", "member"}
missing = need_cols - set(olmoe.column_names)
if missing:
    raise KeyError(
        f"{OLMOE_MEMBER_REPO} split={olmoe_split_name!r} missing {sorted(missing)}. "
        f"Has: {olmoe.column_names}"
    )

olmoe_n_before = len(olmoe)
olmoe = olmoe.filter(lambda ex: ex.get("data_source") not in EXCLUDE_DATA_SOURCES)
print(
    f"{OLMOE_MEMBER_REPO}: {olmoe_n_before} -> {len(olmoe)} rows "
    f"after dropping data_source in {sorted(EXCLUDE_DATA_SOURCES)}"
)


def _olmoe_to_rl(ex):
    ans = ex.get("answer")
    return {
        "data_source": ex.get("data_source"),
        "prompt": ex.get("prompt"),
        "reward_model": {
            "ground_truth": str(ans) if ans is not None else None,
            "style": "rule",
        },
        "member": ex.get("member"),
    }


olmoe_converted = olmoe.map(_olmoe_to_rl, remove_columns=olmoe.column_names)

eurus_rl = load_dataset(EURUS_RL_REPO)
if not isinstance(eurus_rl, DatasetDict) or not {"train", "test"}.issubset(eurus_rl.keys()):
    raise RuntimeError(
        f"{EURUS_RL_REPO} expected to be a DatasetDict with both 'train' and 'test' splits. "
        f"Got: {type(eurus_rl).__name__} keys={list(eurus_rl.keys()) if isinstance(eurus_rl, DatasetDict) else 'n/a'}"
    )

target_cols = ["data_source", "prompt", "reward_model", "member"]


def _prepare_eurus_split(name: str, ds):
    if "data_source" not in ds.column_names:
        raise KeyError(
            f"{EURUS_RL_REPO}[{name!r}] missing 'data_source' column. Has: {ds.column_names}"
        )
    n0 = len(ds)
    ds = ds.filter(lambda ex: ex.get("data_source") not in EXCLUDE_DATA_SOURCES)
    print(
        f"{EURUS_RL_REPO}[{name!r}]: {n0} -> {len(ds)} rows "
        f"after dropping data_source in {sorted(EXCLUDE_DATA_SOURCES)}"
    )
    if "member" not in ds.column_names:
        ds = ds.add_column("member", [0] * len(ds))
    missing = [c for c in target_cols if c not in ds.column_names]
    if missing:
        raise KeyError(
            f"{EURUS_RL_REPO}[{name!r}] missing target columns {missing}. Has: {ds.column_names}"
        )
    drop = [c for c in ds.column_names if c not in target_cols]
    if drop:
        ds = ds.remove_columns(drop)
    return ds


eurus_train = _prepare_eurus_split("train", eurus_rl["train"])
eurus_test = _prepare_eurus_split("test", eurus_rl["test"])

TEST_KEEP_DATA_SOURCES = {"aime", "aime25"}
_test_n_before = len(eurus_test)
eurus_test = eurus_test.filter(lambda ex: ex.get("data_source") in TEST_KEEP_DATA_SOURCES)
print(
    f"{EURUS_RL_REPO}['test']: {_test_n_before} -> {len(eurus_test)} rows "
    f"after keeping only data_source in {sorted(TEST_KEEP_DATA_SOURCES)}"
)
if len(eurus_test) == 0:
    raise RuntimeError(
        f"Test split is empty after restricting to data_source in {sorted(TEST_KEEP_DATA_SOURCES)}."
    )

MIX_SEED = 7

train_combined = concatenate_datasets([olmoe_converted, eurus_train]).shuffle(seed=MIX_SEED)
test_combined = eurus_test.shuffle(seed=MIX_SEED)

print(f"olmoe_member rows (data_source=math, member=1): {len(olmoe_converted)}")
print(f"eurus_rl train rows (member=0): {len(eurus_train)}")
print(f"eurus_rl test  rows (member=0): {len(eurus_test)}")
print(f"train rows: {len(train_combined)} (shuffled, seed={MIX_SEED})")
print(f"test  rows: {len(test_combined)} (shuffled, seed={MIX_SEED})")
print(f"columns: {train_combined.column_names}")
print(f"train member counts: {pd.Series(train_combined['member']).value_counts().to_dict()}")
print(f"test  member counts: {pd.Series(test_combined['member']).value_counts().to_dict()}")

train_sources = pd.Series(train_combined["data_source"]).value_counts()
test_sources = pd.Series(test_combined["data_source"]).value_counts()
unexpected_in_test = set(test_sources.index) - TEST_KEEP_DATA_SOURCES
if unexpected_in_test:
    raise RuntimeError(
        f"test split contains unexpected data_source values: {sorted(unexpected_in_test)}"
    )
print(f"train data_source counts:\n{train_sources}")
print(f"test  data_source counts:\n{test_sources}")

BASE_DIR = Path(".") if Path("filter_benchmark_reasoner.ipynb").exists() else Path("benchmarks")
OUT_DIR = BASE_DIR / "filtered_samples" / "reasoner_rl_train_test"
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_parquet_train = OUT_DIR / "train.parquet"
out_parquet_test = OUT_DIR / "test.parquet"
train_combined.to_pandas().to_parquet(out_parquet_train, index=False)
test_combined.to_pandas().to_parquet(out_parquet_test, index=False)
print(f"Wrote: {out_parquet_train}")
print(f"Wrote: {out_parquet_test}")

display(train_combined.to_pandas().head(3))
display(train_combined.to_pandas().tail(3))
display(test_combined.to_pandas().head(3))


Generating train split: 90 examples [00:00, 3681.73 examples/s]
Filter: 100%|██████████| 90/90 [00:00<00:00, 10831.15 examples/s]


talzoomanzoo/reasoner_member: 90 -> 30 rows after dropping data_source in ['aime26', 'numina_amc_aime']


Map: 100%|██████████| 30/30 [00:00<00:00, 2414.27 examples/s]


talzoomanzoo/eurus_rl_train_test['train']: 1000 -> 970 rows after dropping data_source in ['aime26', 'numina_amc_aime']
talzoomanzoo/eurus_rl_train_test['test']: 100 -> 100 rows after dropping data_source in ['aime26', 'numina_amc_aime']
talzoomanzoo/eurus_rl_train_test['test']: 100 -> 100 rows after keeping only data_source in ['aime', 'aime25']
olmoe_member rows (data_source=math, member=1): 30
eurus_rl train rows (member=0): 970
eurus_rl test  rows (member=0): 100
train rows: 1000 (shuffled, seed=7)
test  rows: 100 (shuffled, seed=7)
columns: ['data_source', 'prompt', 'member', 'reward_model']
train member counts: {0: 970, 1: 30}
test  member counts: {0: 100}
train data_source counts:
olympiads                                     685
cn_contest                                    155
aops_forum                                     79
amc_aime                                       26
inequalities                                   14
olympiads_ref                                   9
TIG

,data_source,prompt,member,reward_model
0,cn_contest,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': 'y=x^{2}-4 x+3', 'style': 'ru..."
1,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '3\frac{1}{3}', 'style': 'rule'}"
2,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '1', 'style': 'rule'}"


,data_source,prompt,member,reward_model
997,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '24', 'style': 'rule'}"
998,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '88', 'style': 'rule'}"
999,olympiads,[{'content': 'Your task is to follow a systema...,0,"{'ground_truth': '66', 'style': 'rule'}"


,data_source,prompt,reward_model,member
0,aime25,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '70', 'style': 'rule'}",0
1,aime25,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '60', 'style': 'rule'}",0
2,aime,[{'content': 'Your task is to follow a systema...,"{'ground_truth': '073', 'style': 'rule'}",0


In [5]:
import importlib
import os
import subprocess
import sys


def _ensure(import_name: str, pip_name: str | None = None) -> None:
    try:
        importlib.import_module(import_name)
    except ImportError:
        pkg = pip_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("huggingface_hub")
_ensure("datasets")
from pathlib import Path
import shutil

from huggingface_hub import HfApi
from datasets import Dataset, DatasetDict, load_dataset

try:
    from huggingface_hub import get_token as _hf_get_token  # huggingface_hub >= 0.19
except ImportError:
    from huggingface_hub import HfFolder  # legacy

    def _hf_get_token():
        return HfFolder.get_token()


HF_REPO_ID_RL = "talzoomanzoo/reasoner_rl_train_test_set"

for _name in ("out_parquet_train", "out_parquet_test"):
    if _name not in globals():
        raise RuntimeError(
            f"Run the mix cell above first; `{_name}` is not defined."
        )

token = _hf_get_token() or os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise RuntimeError(
        "No Hugging Face token found. Run `huggingface-cli login` or set HF_TOKEN/HUGGINGFACE_TOKEN."
    )

api = HfApi(token=token)
api.create_repo(repo_id=HF_REPO_ID_RL, repo_type="dataset", exist_ok=True)

# Workaround for huggingface_hub<=1.12.x bug (#3829, fixed in #3847): commits
# rejected with "Your push was rejected because an LFS pointer pointed to a
# file that does not exist." Two defenses, both safe to apply unconditionally:
#   (1) Clear the in-process Xet write-token cache so we don't reuse a token
#       bound to a previously-deleted backing repo.
#   (2) Force `is_xet_available()` to return False for this process so the
#       upload path negotiates only `["basic", "multipart"]` LFS transfers and
#       skips the Xet code path entirely. `_commit_api._upload_files` reads
#       `constants.HF_HUB_DISABLE_XET` via `is_xet_available()` at runtime,
#       so monkeypatching after import works (no kernel restart needed).
try:
    from huggingface_hub.utils._xet import (
        XET_CONNECTION_INFO_CACHE,
        reset_xet_connection_info_cache_for_repo,
    )
    reset_xet_connection_info_cache_for_repo("dataset", HF_REPO_ID_RL)
    XET_CONNECTION_INFO_CACHE.clear()
    print("Cleared Xet connection info cache.")
except Exception as _exc:
    print(f"(Xet cache reset skipped: {type(_exc).__name__}: {_exc})")

import huggingface_hub.constants as _hf_constants
import huggingface_hub.utils._runtime as _hf_runtime
_hf_constants.HF_HUB_DISABLE_XET = True
os.environ["HF_HUB_DISABLE_XET"] = "1"
print(f"HF_HUB_DISABLE_XET set; is_xet_available()={_hf_runtime.is_xet_available()}")

if "train_combined" in globals() and "test_combined" in globals():
    train_ds = train_combined
    test_ds = test_combined
else:
    train_ds = Dataset.from_parquet(str(out_parquet_train))
    test_ds = Dataset.from_parquet(str(out_parquet_test))

dsd = DatasetDict({"train": train_ds, "test": test_ds})
print(
    f"Pushing DatasetDict to {HF_REPO_ID_RL}: "
    f"train={len(dsd['train'])}, test={len(dsd['test'])}"
)

commit_info = dsd.push_to_hub(
    repo_id=HF_REPO_ID_RL,
    token=token,
    commit_message=(
        "Update train/test splits "
        "(train: olmoe_member data_source=math member=1 + eurus_rl_train_test[train] "
        "minus {numina_amc_aime, aime26}; "
        "test: eurus_rl_train_test[test] restricted to data_source in {aime, aime25})"
    ),
)
print(f"Pushed. commit_info={commit_info}")
print(f"Load with: load_dataset('{HF_REPO_ID_RL}')")

# Local copy for GENERALREASONER eval workflow (see eval_scripts/GENERALREASONER/)
_base = Path(".") if Path("filter_benchmark_reasoner.ipynb").exists() else Path("benchmarks")
_generalreasoner_dir = _base / "GENERALREASONER"
_generalreasoner_dir.mkdir(parents=True, exist_ok=True)
_reasoner_member_path = _generalreasoner_dir / "reasoner_member.parquet"

if "combined" in globals():
    combined.to_parquet(_reasoner_member_path, index=False)
elif "out_parquet" in globals():
    shutil.copy2(out_parquet, _reasoner_member_path)
else:
    _member_ds = load_dataset("talzoomanzoo/reasoner_member", split="train")
    try:
        _member_ds.to_parquet(str(_reasoner_member_path))
    except Exception:
        _member_ds.to_pandas().to_parquet(_reasoner_member_path, index=False)

print(f"Wrote: {_reasoner_member_path.resolve()}")


Cleared Xet connection info cache.
HF_HUB_DISABLE_XET set; is_xet_available()=False
Pushing DatasetDict to talzoomanzoo/reasoner_rl_train_test_set: train=1000, test=100


Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]


Pushed. commit_info=https://huggingface.co/datasets/talzoomanzoo/reasoner_rl_train_test_set/commit/93ffe4719d020b7d47a71dcf5919ba7f6bb2c1d6
Load with: load_dataset('talzoomanzoo/reasoner_rl_train_test_set')
Wrote: /scratch2/mjgwak/rl-data-contamination-mj/benchmarks/GENERALREASONER/reasoner_member.parquet
